# Sửa lemma riding/rides trong bank — CPU

**Accelerator None, Internet ON. Add Input duy nhất `vuthetam/mscoco-2014`.**
Không cần Dataset126, visual-cache, detector, API token hoặc GPU.
Dùng cùng POS/model đã kiểm; chỉ chạy POS lại trên captionTRAIN chứa riding/rides,
không chạy lại toàn566747caption. Đếm token thực sự chuyển rid→ride dưới điều kiện
VERB +riding/rides +lemma cũ rid. Không đổi rid/ridding thật, noun, captiontarget, ROI.

Xuất **bank_words_corrected.json mới**, giữ nguyên bankgốc và Output126.
Save & Run All; gửi `csa_bank_lemma_correction_review.zip`.
Chưa mã hóaCLIPtext/huấn luyện; hạn worker30phút là cap, không phải ETA.


In [ ]:
import hashlib, json, os, queue, signal, subprocess, sys, tempfile, threading, time, traceback, urllib.request, zipfile
from pathlib import Path
OUTPUT=Path('/kaggle/working/csa_bank_lemma_correction')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
if OUTPUT.exists(): raise RuntimeError('Preserve previous review ZIP; use a fresh CPU session.')
OUTPUT.mkdir(parents=True)
SOURCE=Path(tempfile.mkdtemp(prefix='csa-lemma-correction-'))
LOG=OUTPUT/'notebook.log'
BANK_URL='https://raw.githubusercontent.com/Supzxjee/Image_Captioning/28361ce1ccf8ccd71cf9141e7f39b49090f7e6f1/experiments/csa_paper/real_preparation_review_text/bank_words.json'
BANK_SHA='450fc969e0842ca3270d1f899b7e8fe594639260644ad5815a69430fe48264b1'
MODULES={'lemma_correction.py': '"""Narrow, evidence-backed metadata rule; no model imports."""\nfrom collections import Counter\n\n\ndef corrected_lemma(surface, pos, original_lemma):\n    if pos == \'VERB\' and surface.lower() in (\'riding\', \'rides\') and original_lemma.lower() == \'rid\':\n        return \'ride\'\n    return original_lemma\n\n\ndef corrected_bank(original, moved_count):\n    if not isinstance(moved_count, int) or moved_count <= 0 or original[\'minimum_count\'] != 5:\n        raise ValueError(\'Positive measured count and original min5 policy required\')\n    before = Counter({row[\'word\']: row[\'count\'] for row in original[\'verb\']})\n    if before[\'rid\'] < moved_count or not before[\'ride\']:\n        raise ValueError(\'Correction exceeds baseline counts or baseline ride absent\')\n    after = before.copy()\n    after[\'rid\'] -= moved_count\n    after[\'ride\'] += moved_count\n    if sum(after.values()) != sum(before.values()):\n        raise ValueError(\'Count mass changed\')\n    result = dict(original)\n    result[\'verb\'] = [dict(word=w, count=after[w]) for w in sorted(after, key=lambda w: (-after[w], w)) if after[w] >= 5]\n    result[\'lemma_amendment\'] = dict(version=\'verb_riding_rides_rid_to_ride_v1\',\n        surface_forms=[\'riding\', \'rides\'], pos=\'VERB\', original_lemma=\'rid\', corrected_lemma=\'ride\',\n        moved_tokens=moved_count, other_verbs_unchanged=True, nouns_unchanged=True)\n    return result, dict(before_rid=before[\'rid\'], before_ride=before[\'ride\'],\n        after_rid=after[\'rid\'], after_ride=after[\'ride\'], moved_tokens=moved_count,\n        below_minimum_omitted_mass=sum(n for n in after.values() if n < 5))\n', 'run_lemma_correction.py': '"""USER-RUN CPU subset recount; preserve original words, captions and ROI126."""\nimport argparse\nfrom collections import Counter\nimport hashlib\nimport importlib.metadata\nimport json\nfrom pathlib import Path\nimport re\nimport time\nimport traceback\n\nfrom lemma_correction import corrected_lemma, corrected_bank\n\nDATA_SHA = \'2fd999220673258012acfb411a4e7e66af7d488050b2519b0badcc49b7600b8d\'\nBANK_SHA = \'450fc969e0842ca3270d1f899b7e8fe594639260644ad5815a69430fe48264b1\'\n\n\ndef sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef save(path, value):\n    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False)+\'\\n\', encoding=\'utf-8\')\n\n\ndef run(dataset_path, out, report):\n    import spacy\n    here = Path(__file__).parent\n    baseline = here/\'original_bank_words.json\'\n    if sha(dataset_path) != DATA_SHA or sha(baseline) != BANK_SHA:\n        raise ValueError(\'Original dataset/bank checksum differs\')\n    original = json.loads(baseline.read_text(encoding=\'utf-8\'))\n    environment = json.loads((here/\'source_environment.json\').read_text(encoding=\'utf-8\'))\n    versions = {n: importlib.metadata.version(n) for n in (\'spacy\', \'thinc\', \'numpy\', \'en-core-web-sm\')}\n    if versions != environment[\'versions\']:\n        raise ValueError(\'POS versions differ from126\')\n    spacy.require_cpu()\n    nlp = spacy.load(\'en_core_web_sm\', exclude=[\'parser\', \'ner\'])\n    if set(nlp.pipe_names) != {\'tok2vec\', \'tagger\', \'attribute_ruler\', \'lemmatizer\'}:\n        raise ValueError(\'POS pipeline differs\')\n    model_hashes = {str(p.relative_to(nlp.path)): sha(p) for p in sorted(Path(nlp.path).rglob(\'*\')) if p.is_file()}\n    if model_hashes != environment[\'pos_model_files_sha256\']:\n        raise ValueError(\'POS model data differs from126\')\n    dataset = json.loads(Path(dataset_path).read_text(encoding=\'utf-8\'))\n    pattern = re.compile(r\'\\b(?:riding|rides)\\b\', re.I)\n    report[\'train_caption_events_scanned\'] = 0\n    def selected():\n        for image in dataset[\'images\']:\n            if image[\'split\'] not in (\'train\', \'restval\'):\n                continue\n            for sentence_index, sentence in enumerate(image[\'sentences\']):\n                report[\'train_caption_events_scanned\'] += 1\n                if pattern.search(sentence[\'raw\']):\n                    yield sentence[\'raw\'], dict(coco_id=image[\'cocoid\'], filename=image[\'filename\'],\n                        source_split=image[\'split\'], sentence_index=sentence_index, caption=sentence[\'raw\'])\n    moved = 0\n    documents = 0\n    by_surface = Counter()\n    example_counts = Counter()\n    examples = []\n    ledger = hashlib.sha256()\n    for number, (doc, context) in enumerate(nlp.pipe(selected(), as_tuples=True, batch_size=256, n_process=1), 1):\n        documents = number\n        for token_index, token in enumerate(doc):\n            new = corrected_lemma(token.text, token.pos_, token.lemma_)\n            if new == token.lemma_:\n                continue\n            if token.pos_ != \'VERB\' or token.text.lower() not in (\'riding\', \'rides\') or token.lemma_.lower() != \'rid\' or new != \'ride\':\n                raise ValueError(\'Change outside the approved narrow rule\')\n            moved += 1\n            by_surface[token.text.lower()] += 1\n            record = dict(context, token_index=token_index, surface=token.text, pos=token.pos_,\n                          tag=token.tag_, original_lemma=token.lemma_, corrected_lemma=new)\n            ledger.update((json.dumps(record, sort_keys=True, ensure_ascii=False, separators=(\',\', \':\'))+\'\\n\').encode())\n            if example_counts[token.text.lower()] < 10:\n                examples.append(record)\n                example_counts[token.text.lower()] += 1\n        if number % 1000 == 0:\n            print(\'Affected TRAIN captions processed:\', number, \'; redirected tokens:\', moved, flush=True)\n    if report[\'train_caption_events_scanned\'] != 566747:\n        raise ValueError(\'TRAIN caption coverage differs\')\n    corrected, delta = corrected_bank(original, moved)\n    if corrected[\'noun\'] != original[\'noun\'] or sha(baseline) != BANK_SHA:\n        raise ValueError(\'Original nouns or bank changed\')\n    corrected[\'lemma_amendment\'][\'source_bank_sha256\'] = BANK_SHA\n    save(out/\'bank_words_corrected.json\', corrected)\n    save(out/\'correction_examples.json\', examples)\n    report.update(status=\'BANK_LEMMA_CORRECTION_COMPLETE\', affected_documents_pos_processed=documents,\n        delta=delta, redirected_by_surface=dict(by_surface), full_change_ledger_sha256=ledger.hexdigest(),\n        versions=versions, model_files_matched=len(model_hashes), noun_words=len(corrected[\'noun\']),\n        verb_words=len(corrected[\'verb\']), examples_saved=len(examples))\n    save(out/\'correction_receipt.json\', dict(status=report[\'status\'],\n        amendment=\'VERB + riding/rides + original rid -> ride; everything else unchanged\',\n        source_dataset_sha256=DATA_SHA, source_bank_sha256=BANK_SHA, preserved_roi_source_version=126,\n        corrected_bank_sha256=sha(out/\'bank_words_corrected.json\'), delta=delta,\n        nouns_unchanged=True, caption_targets_changed=False, roi_modified=False,\n        semantic_embeddings_created=False, caption_model_executed=False))\n\n\ndef main():\n    parser = argparse.ArgumentParser(); parser.add_argument(\'--dataset-json\', required=True); parser.add_argument(\'--output\', required=True)\n    args = parser.parse_args(); out = Path(args.output); out.mkdir(parents=True, exist_ok=True)\n    report = dict(status=\'INCOMPLETE\', original_bank_modified=False, roi_modified=False,\n                  caption_targets_changed=False, semantic_embeddings_created=False,\n                  caption_model_executed=False, gpu_used=False, dataset_created=False)\n    start = time.monotonic()\n    try:\n        run(Path(args.dataset_json), out, report)\n    except BaseException:\n        report[\'error\'] = traceback.format_exc()\n        raise\n    finally:\n        report[\'seconds\'] = time.monotonic()-start\n        save(out/\'summary.json\', report)\n\n\nif __name__ == \'__main__\':\n    main()\n'}
MODULE_HASHES={'lemma_correction.py': '538de6980996762de4333f96d1f56fb203318c210f68046e87e5e2e1fbfa8dc1', 'run_lemma_correction.py': '3bb7bf49db2a70388574cc5e55f6e57c4de4f1d0689ed43364e9f99ca22187bb'}
ENVIRONMENT='{"pos_model_files_sha256": {"LICENSE": "3933c176979b68bc6d0bcc902c7d6c130f1d127f476f17ba5cdba8d99cfd0012", "LICENSES_SOURCES": "20d9dfacd215112251f15341edd1a47b1f98bf393a212f10f194ffdc7ee98a90", "README.md": "91b0ed1742ea0803034b4d769c4536e88d9d4da7808d833f2a804b6c54aae220", "accuracy.json": "e7572d7f2f3c18e9921471f9d0b3424f8cf7f1a909ae484fa10b780a13f4c676", "attribute_ruler/patterns": "f8b97f7ebef9b8536bba5f0fc1e73af6183563e96e2325bcbb021b8c6dbc6683", "config.cfg": "65c545ca2da5bd4b87b22d4917318605360b37dee6ed6b18516a036f87319f79", "lemmatizer/lookups/lookups.bin": "eb64f40c0f8396d1762730c0ddf4dad2a52d138f5a389f71a1a1d088173b7737", "meta.json": "7456349002fa8cf31111051bd37fdbea67a1b7f7a0a60ce235466f98a6758125", "ner/cfg": "a7172edadafba9f472e9ac0f2660eec04b6405e471be9e20267b79c67288d22d", "ner/model": "cd93e9ca9126c69f1d10a1039ef67c3d9ec1223b44d079856705970f0f19ff4e", "ner/moves": "d82b13473ebd33e0396004dd22f00c71fe3c1251742bf1f7c4554f23fd3b0af4", "parser/cfg": "e2a240ef01f95e58f5dd4c9d6e90d61f597b8d998966570d3009dd81e8879780", "parser/model": "e0098910535a863d430082ece57455d5fa071cd4ca3a0054e80582076c752b1e", "parser/moves": "ec41318c32eb7f9e4d8f050b2e8466ee67954d2d3455c1aec73abf46c82dd67f", "senter/cfg": "2a4db4a729b6469da1f87d71cc7cd3ed2a73360cc85326821e850836c4671103", "senter/model": "b69e7174a0b6307739adec3c8053ac45fdbad7171c4d3d8bfcd9713ecbc01f8c", "tagger/cfg": "d484c3d9391f37bf9c727aa2eb4c477a24cb103020f95e52dc37842818b50118", "tagger/model": "1ec3d93f38cebe172f2b5c89d84be72856ce42c8f1a64f1699f1d98a771f36b7", "tok2vec/cfg": "f8a5a26e3056eb6fb06deeb3dbccfd88ae74900200c98c70b5966bbb7ec9d4de", "tok2vec/model": "e84fc06eb319c94d28e460fc334e292120b01f18baa5dc8b50c977459820a090", "tokenizer": "b014e8bba4958b120af2d0c1c63eabb7c00379f2bacaf10df7c5325efd2ea467", "vocab/key2row": "76be8b528d0075f7aae98d6fa57a6d3c83ae480a8469e668d7b0af968995ac71", "vocab/lookups.bin": "fce9c883c56165f29573cc938c2a1c9d417ac61bd8f56b671dd5f7996de70682", "vocab/strings.json": "b1966d1f07a05b68576df07f43f40203b4e11124ad82cc839e8312ad8d7fdae7", "vocab/vectors": "14772b683e726436d5948ad3fff2b43d036ef2ebbe3458aafed6004e05a40706", "vocab/vectors.cfg": "ff4359091952c8cd16f1f0482f5770fb82d1707368d5cca3c46aa501f552e3c5"}, "versions": {"en-core-web-sm": "3.8.0", "numpy": "2.1.3", "spacy": "3.8.11", "thinc": "8.3.11"}}'

for name, source in MODULES.items():
    payload=source.encode('utf-8')
    if hashlib.sha256(payload).hexdigest()!=MODULE_HASHES[name]: raise RuntimeError('Module SHA differs: '+name)
    (SOURCE/name).write_bytes(payload)
(SOURCE/'source_environment.json').write_text(ENVIRONMENT,encoding='utf-8')
(OUTPUT/'source_receipt.json').write_text(json.dumps(dict(module_sha256=MODULE_HASHES,
    original_bank_url=BANK_URL, original_bank_sha256=BANK_SHA, preserved_roi_source_version=126,
    source_environment_sha256=hashlib.sha256(ENVIRONMENT.encode()).hexdigest()),indent=2),encoding='utf-8')
print('CPU subset recount only; original bank and ROI126 remain unchanged.',flush=True)


In [ ]:
def run(args, timeout=1800):
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('Smoke active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('Smoke wall cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete = False
try:
    if not COCO_JSON.is_file(): raise FileNotFoundError('Add Input vuthetam/mscoco-2014 and check COCO_JSON')
    payload=urllib.request.urlopen(BANK_URL,timeout=120).read()
    if hashlib.sha256(payload).hexdigest()!=BANK_SHA: raise RuntimeError('Original126bank SHA differs')
    (SOURCE/'original_bank_words.json').write_bytes(payload)
    import numpy as np
    os.environ['OMP_NUM_THREADS'] = '2'
    os.environ['OPENBLAS_NUM_THREADS'] = '2'
    os.environ['CUDA_VISIBLE_DEVICES'] = ''
    venv = SOURCE/'pos-env'
    run([sys.executable, '-m', 'venv', '--without-pip', '--system-site-packages', venv], timeout=180)
    PYTHON = venv/'bin/python'
    if '--python' not in subprocess.check_output([sys.executable, '-m', 'pip', '--help'], text=True):
        raise RuntimeError('Base pip must support --python (pip>=22.3); do not install into global environment.')
    PIP = [sys.executable, '-m', 'pip', '--python', str(PYTHON)]
    constraint = SOURCE/'constraints.txt'
    constraint.write_text('numpy==2.1.3'+chr(10), encoding='utf-8')
    run(PIP + ['install', '--disable-pip-version-check', '--only-binary=:all:', '-c', constraint, 'spacy==3.8.11', 'thinc==8.3.11'], timeout=900)
    wheel = SOURCE/'en_core_web_sm-3.8.0-py3-none-any.whl'
    payload = urllib.request.urlopen('https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl', timeout=180).read()
    if hashlib.sha256(payload).hexdigest() != '1932429db727d4bff3deed6b34cfc05df17794f4a52eeb26cf8928f7c1a0fb85': raise RuntimeError('Official POS model wheel SHA differs')
    wheel.write_bytes(payload)
    run(PIP + ['install', '--no-deps', '--force-reinstall', wheel], timeout=180)
    (OUTPUT/'software_freeze.txt').write_text(subprocess.check_output(PIP + ['freeze', '--all'], text=True), encoding='utf-8')
    run([PYTHON,'-u',SOURCE/'run_lemma_correction.py','--dataset-json',COCO_JSON,'--output',OUTPUT],timeout=1800)
    summary=json.loads((OUTPUT/'summary.json').read_text(encoding='utf-8'))
    complete=summary.get('status')=='BANK_LEMMA_CORRECTION_COMPLETE'
    if not complete: raise RuntimeError('Correction incomplete')
    receipt=json.loads((OUTPUT/'correction_receipt.json').read_text(encoding='utf-8'))
    if hashlib.sha256((OUTPUT/'bank_words_corrected.json').read_bytes()).hexdigest()!=receipt['corrected_bank_sha256']: raise RuntimeError('Corrected bank SHA differs')
    print(json.dumps(summary,ensure_ascii=False,indent=2),flush=True)
except BaseException:
    (OUTPUT/'notebook_failure.txt').write_text(traceback.format_exc(),encoding='utf-8')
    raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete,original_bank_modified=False,
        roi_modified=False,caption_targets_changed=False,dataset_created=False)),encoding='utf-8')
    archive=OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUTPUT.iterdir()):
            if p.is_file(): z.write(p,OUTPUT.name+'/'+p.name)
    print('SEND REVIEW ZIP:',archive,flush=True)
    print('CORRECTION COMPLETE; new bank file, original126 preserved.' if complete else 'INCOMPLETE: send ZIP/log.',flush=True)
